In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

In [13]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))

from src.data_processing import create_features

# Load raw data
df = pd.read_csv("../data/raw/data.csv")
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])

# Feature engineering (Task 3)
df = create_features(df)

# Load RFM (Task 4 output)
rfm = pd.read_csv("../data/raw/data.csv")  # fallback if you didn't save rfm

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

# high-risk cluster = 0 (from your Task 4 result)
rfm["is_high_risk"] = (rfm["Cluster"] == 0).astype(int)

In [8]:
df = df.merge(
    rfm[["CustomerId", "is_high_risk"]],
    on="CustomerId",
    how="left"
)

df["is_high_risk"] = df["is_high_risk"].fillna(0)

In [11]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.data_processing import create_features

df = pd.read_csv("../data/raw/data.csv")

df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])

df = create_features(df)

In [14]:
snapshot_date = df["TransactionStartTime"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerId").agg(
    Recency=("TransactionStartTime", lambda x: (snapshot_date - x.max()).days),
    Frequency=("TransactionId", "count"),
    Monetary=("Amount", "sum")
).reset_index()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

# same logic we used before
high_risk_cluster = 0
rfm["is_high_risk"] = (rfm["Cluster"] == high_risk_cluster).astype(int)

In [15]:
df = df.merge(
    rfm[["CustomerId", "is_high_risk"]],
    on="CustomerId",
    how="left"
)

df["is_high_risk"] = df["is_high_risk"].fillna(0)

In [16]:
print(df["is_high_risk"].value_counts())

is_high_risk
0    84653
1    11009
Name: count, dtype: int64


In [ ]:
df = df.merge(
    rfm[["CustomerId", "is_high_risk"]],
    on="CustomerId",
    how="left"
)

df["is_high_risk"] = df["is_high_risk"].fillna(0)

In [ ]:
rfm = pd.read_csv("../data/processed/rfm.csv")

df = df.merge(
    rfm[["CustomerId", "is_high_risk"]],
    on="CustomerId",
    how="left"
)

df["is_high_risk"] = df["is_high_risk"].fillna(0)

In [18]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/data.csv")

# must run feature engineering FIRST
df = create_features(df)

In [19]:
df["is_high_risk"] = (df["FraudResult"] == 1).astype(int)

In [20]:
features = [
    "Amount",
    "Value",
    "TotalTransactionAmount",
    "AverageTransactionAmount",
    "TransactionCount",
    "StdTransactionAmount",
    "TransactionHour",
    "TransactionDay",
    "TransactionMonth",
    "TransactionYear"
]

In [21]:
missing = [c for c in features if c not in df.columns]
if missing:
    raise ValueError(f"Missing features: {missing}")

In [22]:
X = df[features]
y = df["is_high_risk"]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
df = df.merge(
    rfm[["CustomerId", "is_high_risk"]],
    on="CustomerId",
    how="left"
)

df["is_high_risk"] = df["is_high_risk"].fillna(0)

In [24]:
snapshot_date = df["TransactionStartTime"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerId").agg(
    Recency=("TransactionStartTime", lambda x: (snapshot_date - x.max()).days),
    Frequency=("TransactionId", "count"),
    Monetary=("Amount", "sum")
).reset_index()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

# same logic we used before
high_risk_cluster = 0
rfm["is_high_risk"] = (rfm["Cluster"] == high_risk_cluster).astype(int)

In [30]:
customer_features = df.groupby("CustomerId").agg(
    TotalTransactionAmount=("Amount", "sum"),
    AverageTransactionAmount=("Amount", "mean"),
    TransactionCount=("TransactionId", "count"),
    StdTransactionAmount=("Amount", "std")
).reset_index()

print(customer_features.head())
print(customer_features.columns)

        CustomerId  TotalTransactionAmount  AverageTransactionAmount  \
0     CustomerId_1                -10000.0             -10000.000000   
1    CustomerId_10                -10000.0             -10000.000000   
2  CustomerId_1001                 20000.0               4000.000000   
3  CustomerId_1002                  4225.0                384.090909   
4  CustomerId_1003                 20000.0               3333.333333   

   TransactionCount  StdTransactionAmount  
0                 1                   NaN  
1                 1                   NaN  
2                 5           6558.963333  
3                11            560.498966  
4                 6           6030.478146  
Index(['CustomerId', 'TotalTransactionAmount', 'AverageTransactionAmount',
       'TransactionCount', 'StdTransactionAmount'],
      dtype='str')


In [31]:
df = pd.read_csv("../data/raw/data.csv")

df = create_features(df)

print(df.columns)

Index(['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId',
       'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId',
       'ProductCategory', 'ChannelId', 'Amount', 'Value',
       'TransactionStartTime', 'PricingStrategy', 'FraudResult',
       'TotalTransactionAmount', 'AverageTransactionAmount',
       'TransactionCount', 'StdTransactionAmount', 'TransactionHour',
       'TransactionDay', 'TransactionMonth', 'TransactionYear'],
      dtype='str')


In [32]:
features = [
    "Amount",
    "Value",
    "TotalTransactionAmount",
    "AverageTransactionAmount",
    "TransactionCount",
    "StdTransactionAmount",
    "TransactionHour",
    "TransactionDay",
    "TransactionMonth",
    "TransactionYear"
]

In [37]:
features = [
    "Amount",
    "Value",
    "TotalTransactionAmount",
    "AverageTransactionAmount",
    "TransactionCount",
    "StdTransactionAmount",
    "TransactionHour",
    "TransactionDay",
    "TransactionMonth",
    "TransactionYear"
]

X = df[features]
y = df["FraudResult"]

print(X.shape)
print(y.shape)

(95662, 10)
(95662,)


In [39]:
X = df[features]
y = df["FraudResult"]

print(X.shape)
print(y.shape)

(95662, 10)
(95662,)


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (76529, 10)
Test shape: (19133, 10)


In [41]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [42]:
y_pred = model.predict(X_test)

In [43]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9980661684001464
Precision: 0.5357142857142857
Recall: 0.38461538461538464
F1 Score: 0.44776119402985076

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19094
           1       0.54      0.38      0.45        39

    accuracy                           1.00     19133
   macro avg       0.77      0.69      0.72     19133
weighted avg       1.00      1.00      1.00     19133



In [45]:
import joblib

joblib.dump(model, "../models/logistic_regression.pkl")

print("Model saved!")

Model saved!
